# 02 - Regional category heatmaps (100-mi)

% deviation from SARIMAX baseline, days x 6 categories. **6 panels = 3 flows (within/inflow/outflow) x 2 storms**,
matching the existing 50-mi `figure2` layout. Plot-only: reads `00`'s cached baselines, no re-fit.

Color scale is symmetric (RdBu, centred 0) and **shared across both hurricanes within each flow**
(within/inflow/outflow get their own scale since magnitudes differ).

Outputs (each a separate file) -> `results/npj_100mi/figure2_heatmap_{flow}_{hurricane}.{pdf,png}`.

In [1]:
import pathlib, warnings
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.dates import DateFormatter, DayLocator
warnings.filterwarnings('ignore')

mpl.rcParams.update({
    'font.family':'sans-serif','font.sans-serif':['Arial','Helvetica','DejaVu Sans'],
    'font.size':8,'axes.titlesize':8,'axes.labelsize':8,'xtick.labelsize':7,'ytick.labelsize':7,
    'legend.fontsize':7,'axes.linewidth':0.6,'axes.spines.top':False,'axes.spines.right':False,
    'savefig.dpi':300,'savefig.bbox':'tight','pdf.fonttype':42,'ps.fonttype':42,
})

In [2]:
# Path resolution -> the 100-mi regional data produced by 00_regional_flows_100mi.ipynb
def find_project_root(start):
    sentinel = pathlib.Path('results')/'npj_100mi'/'regional_data'/'regional_metrics_summary_100mi.csv'
    for p in [start, *start.parents]:
        if (p/sentinel).exists():
            return p
    raise FileNotFoundError(f'Run 00 first; cannot find {sentinel} from {start}')

PROJECT_ROOT = find_project_root(pathlib.Path.cwd())
RESULTS  = PROJECT_ROOT/'results'
REGIONAL = RESULTS/'npj_100mi'/'regional_data'          # 100-mi baselines + metrics (from 00)
METRICS_CSV = REGIONAL/'regional_metrics_summary_100mi.csv'
OUT_DIR  = RESULTS/'npj_100mi'                            # flat figure outputs
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('REGIONAL exists:', REGIONAL.exists(), '| METRICS exists:', METRICS_CSV.exists())

REGIONAL exists: True | METRICS exists: True


In [3]:
HURRICANES = {
    'helene': {'label':'Helene','landing':pd.Timestamp('2024-09-26'),'color':'#1f77b4'},
    'milton': {'label':'Milton','landing':pd.Timestamp('2024-10-09'),'color':'#d62728'},
}
CATEGORIES = ['Travel','Work & Professional','Health','Education','Retail & Leisure','Urban Government']
CATEGORY_COLORS = {'Travel':'#0072B2','Work & Professional':'#E69F00','Health':'#009E73',
    'Education':'#CC79A7','Retail & Leisure':'#56B4E9','Urban Government':'#D55E00'}
FLOW_COLORS = {'within':'#2b8a3e','inflow':'#1971c2','outflow':'#c92a2a'}
FLOW_LABELS = {'within':'Within','inflow':'Inflow','outflow':'Outflow'}

def category_to_filename(c): return c.replace(' & ','_and_').replace(' ','_')
def save_panel(fig, stem):
    fig.savefig(OUT_DIR/f'{stem}.pdf', bbox_inches='tight')
    fig.savefig(OUT_DIR/f'{stem}.png', dpi=300, bbox_inches='tight')
    print('  saved ->', stem, '(.pdf/.png)')

PRE_DAYS, POST_DAYS = 7, 14
FLOWS = ['within','inflow','outflow']

In [4]:
# Load % deviation matrices (categories x days) for every (flow, hurricane) from cached baselines
def load_rd_matrix(hkey, flow):
    landing = HURRICANES[hkey]['landing']
    win_start, win_end = landing - pd.Timedelta(days=PRE_DAYS), landing + pd.Timedelta(days=POST_DAYS)
    rows = {}
    for cat in CATEGORIES:
        bl = pd.read_csv(REGIONAL/hkey/f'baseline_{flow}_{category_to_filename(cat)}.csv',
                         index_col=0, parse_dates=True).loc[win_start:win_end]
        rows[cat] = (bl['y_true'] - bl['y_pred'])/bl['y_pred']*100.0
    df = pd.DataFrame(rows).T
    df.index.name='category'; df.columns.name='date'
    return df

RD = {(f,h): load_rd_matrix(h,f) for f in FLOWS for h in HURRICANES}
for (f,h),df in RD.items():
    print(f'{f:7s} {h:7s} shape={df.shape} range=[{df.values.min():+.1f}%, {df.values.max():+.1f}%]')

within  helene  shape=(6, 21) range=[-13.2%, +14.7%]
within  milton  shape=(6, 21) range=[-44.7%, +20.1%]
inflow  helene  shape=(6, 21) range=[-27.2%, +20.9%]
inflow  milton  shape=(6, 21) range=[-60.3%, +25.0%]
outflow helene  shape=(6, 21) range=[-20.0%, +21.3%]
outflow milton  shape=(6, 21) range=[-19.0%, +46.2%]


In [6]:
# Per-flow symmetric color scale (shared across both hurricanes within a flow)
VMAX = {}
for f in FLOWS:
    m = max(abs(RD[(f,h)].values).max() for h in HURRICANES)
    VMAX[f] = float(np.ceil(m/10.0)*10.0)
print('per-flow color scale (+/- %):', VMAX)

def panel_heatmap(ax, flow, hkey):
    rd = RD[(flow,hkey)]
    dates = pd.to_datetime(rd.columns); landing = HURRICANES[hkey]['landing']
    norm = TwoSlopeNorm(vmin=-VMAX[flow], vcenter=0.0, vmax=VMAX[flow])
    im = ax.imshow(rd.values, aspect='auto', cmap='RdBu', norm=norm, interpolation='nearest',
        extent=[mpl.dates.date2num(dates[0])-0.5, mpl.dates.date2num(dates[-1])+0.5,
                len(rd.index)-0.5, -0.5])
    ax.xaxis_date()
    ax.axvline(landing, color='black', linestyle='--', linewidth=0.8)
    ax.text(landing, len(rd.index)-0.55, 'landfall', ha='right', va='bottom',
            rotation=90, fontsize=6, color='#444')
    ax.set_yticks(range(len(rd.index)))
    if hkey == 'milton':
        ax.set_yticklabels([])                  # no category labels for Milton (shares Helene's)
    else:
        ax.set_yticklabels(rd.index)
        for tick,cat in zip(ax.get_yticklabels(), rd.index): tick.set_color(CATEGORY_COLORS[cat])
    ax.xaxis.set_major_locator(DayLocator(interval=3)); ax.xaxis.set_major_formatter(DateFormatter('%b %d'))
    ax.tick_params(axis='x', rotation=0); ax.set_xlabel('Date (2024)')
    ax.set_title(f"{HURRICANES[hkey]['label']} . {FLOW_LABELS[flow]} deviation (100 mi)",
                 loc='left', fontsize=8, color='#333', pad=4)
    if hkey != 'helene':                        # colorbar on Milton only (shared scale within flow)
        cbar = ax.figure.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
        cbar.set_label('% deviation from baseline', fontsize=7); cbar.ax.tick_params(labelsize=6)
    return im

for flow in FLOWS:
    for hkey in HURRICANES:
        fig, ax = plt.subplots(figsize=(3.5,1.8))
        panel_heatmap(ax, flow, hkey)
        save_panel(fig, f'figure2_heatmap_{flow}_{hkey}')
        plt.close(fig)
print('done: 6 heatmap panels')

per-flow color scale (+/- %): {'within': 50.0, 'inflow': 70.0, 'outflow': 50.0}
  saved -> figure2_heatmap_within_helene (.pdf/.png)
  saved -> figure2_heatmap_within_milton (.pdf/.png)
  saved -> figure2_heatmap_inflow_helene (.pdf/.png)
  saved -> figure2_heatmap_inflow_milton (.pdf/.png)
  saved -> figure2_heatmap_outflow_helene (.pdf/.png)
  saved -> figure2_heatmap_outflow_milton (.pdf/.png)
done: 6 heatmap panels
